# Telco Customer Churn — Full ML Lifecycle

**Dataset:** IBM Telco Customer Churn  
**Goal:** Predict churn (binary classification) and convert results into retention actions.

This notebook mirrors `run_pipeline.py` for interactive exploration.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from src.preprocess import load_raw, clean_dataframe, build_feature_frame, engineer_features
from src.train import train_all_models, select_best_model
from src.evaluate import evaluate_classifier, estimate_business_impact
from src.config import TARGET

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

## 1. Problem definition
- **Business problem:** Customers leaving the telecom service reduce recurring revenue.
- **ML task:** Classify each customer as likely to churn or not.
- **Stakeholders:** Retention CRM, product/network, finance.
- **KPIs:** Churn rate, recall, ROC-AUC, estimated retention ROI.

## 2. Load & understand data

In [ ]:
raw = load_raw()
print(raw.shape)
display(raw.head())
display(raw.dtypes.to_frame('dtype'))
print('Churn balance:')
display(raw['Churn'].value_counts(normalize=True))

## 3. Cleaning & preprocessing
- Convert `TotalCharges` to numeric (blank strings → 0 for tenure 0)
- Drop duplicates
- Map target Yes/No → 1/0
- Drop `customerID` before modeling (ID leakage prevention)

In [ ]:
clean = build_feature_frame(clean_dataframe(raw))
featured = engineer_features(clean)
print('Missing values after clean:')
display(featured.isna().sum()[featured.isna().sum() > 0])
display(featured.head())

## 4. EDA

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
sns.countplot(data=featured, x=TARGET, ax=ax)
ax.set_xticklabels(['No Churn','Churn'])
ax.set_title('Target balance')
plt.show()

fig, axes = plt.subplots(1,3, figsize=(14,4))
for ax, col in zip(axes, ['tenure','MonthlyCharges','TotalCharges']):
    sns.kdeplot(data=featured, x=col, hue=TARGET, fill=True, common_norm=False, ax=ax)
plt.tight_layout(); plt.show()

display(featured.groupby('Contract')[TARGET].mean().sort_values(ascending=False).to_frame('churn_rate'))
display(featured.groupby('InternetService')[TARGET].mean().sort_values(ascending=False).to_frame('churn_rate'))

## 5. Feature engineering (already applied above)
- `AvgChargesPerMonth`, `TenureGroup`, `HasStreamingBundle`, `AddonCount`, `IsAutoPay`
- No target leakage: engineered only from known customer attributes

## 6–7. Train models & evaluate

In [ ]:
results = train_all_models(featured)
rows = []
for name, payload in results.items():
    m = evaluate_classifier(payload['pipeline'], payload['X_test'], payload['y_test'])
    rows.append({
        'model': name,
        'cv_roc_auc': payload['cv_roc_auc_mean'],
        'test_roc_auc': m['roc_auc'],
        'precision': m['precision'],
        'recall': m['recall'],
        'f1': m['f1'],
        'accuracy': m['accuracy'],
    })
cmp = pd.DataFrame(rows).sort_values('cv_roc_auc', ascending=False)
display(cmp)

best_name, best_pipe = select_best_model(results)
best_metrics = evaluate_classifier(best_pipe, results[best_name]['X_test'], results[best_name]['y_test'])
print('Best model:', best_name)
print('Confusion matrix:', best_metrics['confusion_matrix'])

## 8. Business impact sketch

In [ ]:
impact = estimate_business_impact(
    results[best_name]['y_test'].values,
    best_metrics['y_prob'],
    avg_monthly_revenue=float(featured['MonthlyCharges'].mean()),
)
display(pd.Series(impact))
Markdown(Path(ROOT/'outputs'/'business_insights.md').read_text(encoding='utf-8') if (ROOT/'outputs'/'business_insights.md').exists() else '_Run `python run_pipeline.py` once to generate saved insights._')